In [0]:
circuit_df = spark.sql("select * from batch4.class.circuits")

display(circuit_df)

In [0]:
race_df = spark.sql("select * from batch4.class.races") \
               .withColumnRenamed("Name","Race_Name") \
               .withColumnRenamed("Date","Race_Date")

display(race_df)

In [0]:
race_circuit_df = race_df.join(circuit_df, race_df.Circuit_ID == circuit_df.CircuitID, "inner") \
                         .selectExpr(["CAST(Race_ID AS INT)", "Year", "CAST(Round AS INT)", "Name", "Location", "Country", "Race_Name", "CAST(Race_Date AS DATE)", "Time"])

display(race_circuit_df)

In [0]:
%sql
create or replace table batch4.class.race_circuit (
    Race_ID int,
    Year string,
    Round int,
    Circuit_Name string,
    Location string,
    Country string,
    Race_Name string,
    Race_Date date,
    Time string
)
USING DELTA

In [0]:
race_circuit_df.write.mode("overwrite").saveAsTable("batch4.class.temp_race_circuit")

In [0]:
%sql
MERGE INTO batch4.class.race_circuit t
USING batch4.class.temp_race_circuit s
ON s.Race_ID = t.Race_ID AND (t.Year <> s.Year OR t.Round <> s.Round OR t.Circuit_Name <> s.Name OR
                              t.Location <> s.Location OR t.Country <> s.Country OR t.Race_Name <> s.Race_Name OR
                              t.Race_Date <> s.Race_Date OR t.Time <> s.Time)

WHEN MATCHED THEN
UPDATE SET t.Year = s.Year,
    t.Round = s.Round,
    t.Circuit_Name = s.Name,
    t.Location = s.Location,
    t.Country = s.Country,
    t.Race_Name = s.Race_Name,
    t.Race_Date = s.Race_Date,
    t.Time = s.Time


In [0]:
%sql
INSERT INTO batch4.class.race_circuit
select s.*
from batch4.class.temp_race_circuit s
LEFT JOIN batch4.class.race_circuit t
ON s.Race_ID = t.Race_ID
where t.Race_ID is null

In [0]:
%sql
select * from batch4.class.race_circuit

In [0]:
%sql
update batch4.class.temp_race_circuit
set Location = 'Melbourne'
where Race_ID = 1

In [0]:
%sql
select * from batch4.class.temp_race_circuit